# **MÓDULO 20 - Projeto de Credit Score - Naive Bayes**


No módulo 17 vocês realizaram a primeira etapa do projeto de crédito de vocês.
Então fizeram o tratamendo dos dados, balancearam as classes, transformaram as variáveis categóricas e separam base de treino e teste.
Nessa aula aplicaremos o algoritmo de naive bayes a base de vocês afim de tentarmos trazer previsões do score de crédito.

**IMPORTANTE:** Não se esqueçam de ao enviar o código de vocês para os tutores, enviarem as bases, pois como cada um de vocês realizou as alterações de tratamento indidualmente o tutor precisa ter acesso aos seus dados individuais.

In [76]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score

Durante a aula nossa variável a ser prevista (churn) continha apenas 2 categorias, a base de vocês contém mais. O Naive Bayes pode ser aplicado para problemas de classificação com múltiplas classes da mesma forma que para problemas de classificação binária. O Naive Bayes é um algoritmo de classificação probabilístico que calcula a probabilidade de uma amostra pertencer a cada classe e seleciona a classe com a maior probabilidade como a previsão final.
Em resumo, o Naive Bayes pode ser aplicado da mesma maneira para problemas de classificação com múltiplas classes, e os mesmos princípios se aplicam em termos de treinamento, avaliação e aplicação do modelo.

# 1) Comece carregando as bases de treino (X e y) e teste (X e y).
Verifique se o número de linhas condiz, se as variáveis estão corretas sendo apenas a de score para y e as demais nas bases de X e por último, se Y está balanceada no teste.

In [77]:
X_train = pd.read_csv('X_train_balanced.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train_balanced.csv')
y_test = pd.read_csv('y_test.csv')

In [79]:
X_train.dtypes

Credit_Score        int64
Idade             float64
Tempo_Credito       int64
Balanco           float64
Qtd_Produtos        int64
Possui_Cartao     float64
Membro_Ativo      float64
Salario_Anual     float64
Genero_encoded      int64
Pais_GERMANY        int64
Pais_SPAIN          int64
dtype: object

In [80]:
print(f"X_train: {X_train.shape[0]} linhas | y_train: {len(y_train)} linhas")
print(f"X_test: {X_test.shape[0]} linahs | y_test: {len(y_test)} linhas")

assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)

X_train: 117950 linhas | y_train: 117950 linhas
X_test: 24927 linahs | y_test: 24927 linhas


In [81]:
print("Colunas em y_train:", y_train.columns.tolist())
print("Colunas em X_train:", X_train.columns.tolist())

Colunas em y_train: ['Churn']
Colunas em X_train: ['Credit_Score', 'Idade', 'Tempo_Credito', 'Balanco', 'Qtd_Produtos', 'Possui_Cartao', 'Membro_Ativo', 'Salario_Anual', 'Genero_encoded', 'Pais_GERMANY', 'Pais_SPAIN']


In [82]:
if 'Credit_Score' in X_train.columns:
    y_train = X_train[['Credit_Score']]
    X_train = X_train.drop(columns=['Credit_Score'])

In [83]:
y_train.columns = ['Credit_Score']
y_test.columns = ['Credit_Score']
print("Variáveis em X_train:", X_train.columns.tolist())
print("Variáveis em y_train:", y_train.columns.tolist())
print("Balanceamento no Teste:", y_test['Credit_Score'].value_counts(normalize=True))

Variáveis em X_train: ['Idade', 'Tempo_Credito', 'Balanco', 'Qtd_Produtos', 'Possui_Cartao', 'Membro_Ativo', 'Salario_Anual', 'Genero_encoded', 'Pais_GERMANY', 'Pais_SPAIN']
Variáveis em y_train: ['Credit_Score']
Balanceamento no Teste: Credit_Score
0.0    0.788262
1.0    0.211738
Name: proportion, dtype: float64


In [84]:
contagem = y_test['Credit_Score'].value_counts()
print("Distribuição absoluta:", contagem)
proporcao = y_test['Credit_Score'].value_counts()
print("Distribuição percentual:", proporcao)

Distribuição absoluta: Credit_Score
0.0    19649
1.0     5278
Name: count, dtype: int64
Distribuição percentual: Credit_Score
0.0    19649
1.0     5278
Name: count, dtype: int64


In [85]:
print("Valores únicos no treino:", y_train.iloc[:, 0].nunique())
print("Valores únicos no teste:", y_test.iloc[:, 0].nunique())

Valores únicos no treino: 460
Valores únicos no teste: 2


In [86]:
limite = y_train['Credit_Score'].median()

y_train['Credit_Score_Class'] = (y_train['Credit_Score'] > limite).astype(int)

print("Novos valores únicos no treino:", y_train['Credit_Score_Class'].nunique())

Novos valores únicos no treino: 2


# 2) Aplique o algoritmo de Naive Bayes aos dados de treinamento.

In [96]:
modelo_nb = GaussianNB()
modelo_nb.fit(X_train, y_train['Credit_Score_Class'])

GaussianNB()

# 3) Faça a avaliação do modelo com os dados de treinamento.
Traga a acurácia, recall e plote a matriz de confusão. Não se esqueça de avaliar com suas palavras o desempenho do modelo, interpretando as métricas.

Dica: Para calcularmos o recall em classificação multi classe precisamos usar o atributo macro:
recall = recall_score(y_train, y_pred_train, average='macro')

In [105]:
y_pred_train = modelo_nb.predict(X_train)
rec_train = recall_score(y_train['Credit_Score_Class'], y_pred_train, average='macro')

print(f"--- TREINO ---")
print(f"Acurácia: {acc_train:.4f}")
print(f"Recall (Macro): {rec_train:.4f}")

--- TREINO ---
Acurácia: 0.0036
Recall (Macro): 0.5084


# 4) Aplique o modelo aos dados de teste e realize a avaliação dos resultados, da mesma forma que fez acima. Não se esqueça de avaliar com as suas palavras e comparar o desempenho da base treino com a teste.

In [106]:
if 'Credit_Score' in X_test.columns:
    X_test = X_test.drop(columns=['Credit_Score'])

In [107]:
X_test_limpo = X_test.drop(columns=['Credit_Score']) if 'Credit_Score' in X_test.columns else X_test
y_pred_test = modelo_nb.predict(X_test_limpo)
rec_test = recall_score(y_test, y_pred_test, average='macro')

print(f"\n--- TESTE ---")
print(f"Acurácia: {acc_test:.4f}")
print(f"Recall (Macro): {rec_test:.4f}")


--- TESTE ---
Acurácia: 0.3536
Recall (Macro): 0.4864


In [108]:
corte = y_train['Credit_Score'].median()
y_train_cat = (y_train['Credit_Score'] > corte).astype(int)

modelo_nb = GaussianNB()
modelo_nb.fit(X_train, y_train_cat)

y_pred_train = modelo_nb.predict(X_train)
y_pred_test = modelo_nb.predict(X_test_limpo) # X_test sem a coluna 'Credit_Score'

print(f"--- TREINO REVISADO ---")
print(f"Acurácia: {accuracy_score(y_train_cat, y_pred_train):.4f}")
print(f"Recall (Macro): {recall_score(y_train_cat, y_pred_train, average='macro'):.4f}")

print(f"\n--- TESTE REVISADO ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Recall (Macro): {recall_score(y_test, y_pred_test, average='macro'):.4f}")

--- TREINO REVISADO ---
Acurácia: 0.5074
Recall (Macro): 0.5084

--- TESTE REVISADO ---
Acurácia: 0.3536
Recall (Macro): 0.4864


In [ ]:
'''
Acurácia e Recall no Treino (~50.8%): Indica que o modelo está classificando corretamente apenas metade dos casos. 
Em um cenário binário, isso sugere que o modelo não conseguiu capturar padrões matemáticos significativos nas 
variáveis preditoras para distinguir entre as classes de score.

Acurácia no Teste (35.36%): Este valor é inferior ao acaso. Ocorre porque o modelo foi treinado em uma base
balanceada (50/50), mas está sendo testado em uma base desbalanceada onde a classe majoritária representa 
quase 79% dos dados.

Underfitting: O modelo é simples demais para a complexidade dos dados de crédito, 
falhando em aprender tanto no treino quanto no teste.
'''

# 5) Descreva com suas palavras o projeto desenvolvido nessa atividade e qual o nosso objetivo principal ao aplicarmos o algoritmo de naive bayes a base de crédito.
Utilize pelo menos 4 linhas.

Dica: Caso você ainda esteja tendo dificuldade em visualizar a aplicação dos projetos e objetivo, consulte seus tutores!

In [ ]:
'''
O projeto consiste na aplicação do algoritmo Naive Bayes para a classificação de risco 
em uma base de dados de Credit Score, utilizando bibliotecas de Data Science como Pandas 
e Scikit-learn. O objetivo principal foi utilizar um modelo probabilístico para prever 
se um cliente pertence a uma determinada categoria de score (ex: 0 ou 1) com base em variáveis
preditoras como idade, salário, balanço bancário e tempo de crédito. Ao aplicarmos 
o Naive Bayes, buscamos um modelo que calcula a probabilidade de cada classe de forma 
independente, servindo como uma linha de base (baseline) para identificar padrões de crédito 
e automatizar a tomada de decisão financeira. Por fim, o projeto visou avaliar a eficácia do 
modelo através de métricas de Acurácia e Recall, garantindo que a análise de risco seja capaz
de identificar corretamente tanto bons quanto maus pagadores mesmo em bases de dados desbalanceadas.
'''